# FEASE Deployment Artifact & Qualitative Testing

This notebook produces a shippable model artifact and checks it qualitatively before release. It covers:

1. Data quality validation against historical baselines.
2. Training with production-style hyperparameters and advanced weighting.
3. Saving the `.fease` artifact alongside a manifest (`manifest.json`) recording provenance.
4. Reloading from disk and verifying bit-exact predictions.
5. Latency benchmarks (single predict, batch predict).
6. Persona-based qualitative spot checks.
7. More-Like-This coherence inspection.

**Prereq.** Run `.venv/bin/maturin develop` first so `rust_fease_recommender` is importable.

In [ ]:
import datetime as dt
import json
import statistics
import tempfile
import time
from collections import Counter
from pathlib import Path

import numpy as np
import polars as pl

import rust_fease_recommender as fease

RNG = np.random.default_rng(seed=7)

# In production, point this at an S3/DBFS path. For demo we use a temp dir.
ARTIFACT_DIR = Path(tempfile.mkdtemp(prefix="fease_artifact_"))
DATA_DIR = Path(tempfile.mkdtemp(prefix="fease_data_"))
print(f"Artifact dir: {ARTIFACT_DIR}")
print(f"Data dir:     {DATA_DIR}")

## 1. Prepare training data

We reuse the generator from the quickstart notebook but at a slightly larger scale, and add optional `event_type` and `days_ago` columns to exercise the advanced weighting path.

In [ ]:
N_USERS, N_ITEMS = 800, 150
GENRES = ["Action", "Comedy", "Drama", "Romance", "SciFi", "Documentary"]
PLANS = ["Free", "Premium"]
DEVICES = ["Mobile", "Web", "Console", "TV"]
REGIONS = ["US", "EMEA", "APAC", "LATAM"]
EVENT_TYPES = ["click", "cart", "purchase"]

PERSONAS = {
    "action":      ("Action", "SciFi"),
    "comedy":      ("Comedy", "Romance"),
    "drama":       ("Drama", "Romance"),
    "scifi":       ("SciFi", "Action"),
    "documentary": ("Documentary", "Drama"),
}

item_ids = [f"I{i:04d}" for i in range(N_ITEMS)]
item_genre = {iid: GENRES[i % len(GENRES)] for i, iid in enumerate(item_ids)}
item_type = {iid: ("movie" if i % 3 == 0 else "episode") for i, iid in enumerate(item_ids)}
item_feat_rows = []
for iid in item_ids:
    item_feat_rows.append((iid, f"genre_{item_genre[iid]}", 1.0))
    item_feat_rows.append((iid, f"type_{item_type[iid]}", 1.0))

user_ids = [f"U{i:05d}" for i in range(N_USERS)]
persona_names = list(PERSONAS.keys())
user_persona = {uid: persona_names[RNG.integers(0, len(persona_names))] for uid in user_ids}
user_plan = {uid: PLANS[RNG.integers(0, len(PLANS))] for uid in user_ids}
user_device = {uid: DEVICES[RNG.integers(0, len(DEVICES))] for uid in user_ids}
user_region = {uid: REGIONS[RNG.integers(0, len(REGIONS))] for uid in user_ids}

user_feat_rows = []
for uid in user_ids:
    user_feat_rows.append((uid, f"plan_{user_plan[uid]}", 1.0))
    user_feat_rows.append((uid, f"device_{user_device[uid]}", 1.0))
    user_feat_rows.append((uid, f"region_{user_region[uid]}", 1.0))

inter_rows = []
for uid in user_ids:
    primary, secondary = PERSONAS[user_persona[uid]]
    weights = np.array([
        4.0 if item_genre[iid] == primary
        else 2.0 if item_genre[iid] == secondary
        else 0.4
        for iid in item_ids
    ])
    probs = weights / weights.sum()
    n_watched = int(RNG.integers(10, 30))
    picks = RNG.choice(item_ids, size=n_watched, replace=False, p=probs)
    for iid in picks:
        inter_rows.append((
            uid,
            iid,
            float(RNG.uniform(1.0, 5.0)),
            EVENT_TYPES[RNG.integers(0, len(EVENT_TYPES))],
            float(RNG.integers(0, 180)),  # days_ago
        ))

df_interactions = pl.DataFrame(
    inter_rows,
    schema=["user_id", "item_id", "value", "event_type", "days_ago"],
    orient="row",
)
df_user_features = pl.DataFrame(user_feat_rows, schema=["user_id", "feature_name", "value"], orient="row")
df_items = pl.DataFrame(item_feat_rows, schema=["item_id", "feature_name", "value"], orient="row")

i_path = DATA_DIR / "interactions.parquet"
u_path = DATA_DIR / "user_features.parquet"
t_path = DATA_DIR / "item_features.parquet"
df_interactions.write_parquet(i_path)
df_user_features.write_parquet(u_path)
df_items.write_parquet(t_path)

print(f"Interactions: {df_interactions.height}  Users: {N_USERS}  Items: {N_ITEMS}")

## 2. Data quality validation

`fease.validate_data` checks current run counts against a rolling baseline. A gate in front of `build_and_train` is the simplest way to prevent a bad upstream join from silently producing a degraded model.

In [ ]:
# Pretend these came from the last 14 daily training runs
historical_users = [780.0, 805.0, 790.0, 810.0, 795.0, 802.0, 798.0,
                    809.0, 788.0, 801.0, 792.0, 807.0, 803.0, 799.0]
historical_items = [148.0, 150.0, 149.0, 151.0, 148.0, 150.0, 149.0,
                    150.0, 148.0, 151.0, 150.0, 149.0, 148.0, 150.0]
historical_inter = [16200.0, 16500.0, 16350.0, 16410.0, 16100.0, 16540.0, 16300.0,
                    16480.0, 16150.0, 16520.0, 16380.0, 16460.0, 16320.0, 16400.0]

distinct_users = df_interactions.select(pl.col("user_id").n_unique()).item()
distinct_items = df_interactions.select(pl.col("item_id").n_unique()).item()
n_inter = df_interactions.height

passed, messages = fease.validate_data(
    historical_users, historical_items, historical_inter,
    current_users=float(distinct_users),
    current_items=float(distinct_items),
    current_interactions=float(n_inter),
)
print(f"Data validation passed={passed}")
for m in messages:
    print(" -", m)
assert passed, "Data validation failed — halt training in prod."

## 3. Train

We turn on the full complement of advanced weighting — event-type weights, exponential temporal decay, sparsity pruning — to show the production-flavored knobs.

In [ ]:
HYPERPARAMS = dict(
    alpha=1.0,
    beta=1.0,
    lambda_=120.0,
    meta_weight=0.0,
    decay_rate=0.005,             # half-life ~ ln(2)/0.005 = 139 days
    sparsity_threshold=0.001,     # prune tiny S-matrix entries
    event_weights={"click": 1.0, "cart": 3.0, "purchase": 5.0},
)

t0 = time.perf_counter()
model = fease.build_and_train(
    interactions_path=str(i_path),
    user_features_path=str(u_path),
    item_features_path=str(t_path),
    **HYPERPARAMS,
)
train_seconds = time.perf_counter() - t0

passed, validation_msgs = model.validate()
print(f"Trained in {train_seconds:.2f}s, validate() passed={passed}")
for m in validation_msgs:
    print(" -", m)
assert passed

## 4. Save artifact + manifest

A deployable artifact is more than just the `.fease` binary. We write a sibling `manifest.json` capturing:

- hyperparameters used
- dataset fingerprints (row counts, distinct counts)
- model dimensions
- training duration
- UTC timestamp

This lets downstream services verify they're loading the right model and gives you something to diff when quality drifts.

In [ ]:
model_path = ARTIFACT_DIR / "model.fease"
manifest_path = ARTIFACT_DIR / "manifest.json"

model.save(str(model_path))

manifest = {
    "schema_version": 1,
    "trained_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "artifact": {
        "path": model_path.name,
        "size_bytes": model_path.stat().st_size,
    },
    "model": {
        "num_items": model.num_items,
        "num_user_features": model.num_user_features,
        "num_item_features": model.num_item_features,
    },
    "hyperparameters": HYPERPARAMS,
    "data": {
        "interactions_rows": df_interactions.height,
        "distinct_users": distinct_users,
        "distinct_items": distinct_items,
        "user_feature_rows": df_user_features.height,
        "item_feature_rows": df_items.height,
    },
    "training": {
        "duration_seconds": round(train_seconds, 3),
        "validation_passed": passed,
        "validation_messages": validation_msgs,
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2))

print(f"Wrote {model_path.name} ({model_path.stat().st_size/1024:.1f} KiB)")
print(f"Wrote {manifest_path.name}")
print(json.dumps(manifest, indent=2))

## 5. Fresh load + equivalence check

Simulate a service startup: load the model from disk into a new handle and confirm predictions match the in-memory model.

In [ ]:
loaded = fease.load_model(str(model_path))
assert loaded.num_items == model.num_items
assert loaded.num_user_features == model.num_user_features
assert loaded.num_item_features == model.num_item_features

sample_uid = user_ids[0]
sample_inter = dict(
    df_interactions.filter(pl.col("user_id") == sample_uid)
    .select(["item_id", "value"]).iter_rows()
)
sample_feats = dict(
    df_user_features.filter(pl.col("user_id") == sample_uid)
    .select(["feature_name", "value"]).iter_rows()
)

orig = model.predict(sample_inter, sample_feats, top_k=10)
fresh = loaded.predict(sample_inter, sample_feats, top_k=10)
for (g1, s1), (g2, s2) in zip(orig, fresh):
    assert g1 == g2 and abs(s1 - s2) < 1e-10
print("Reloaded artifact produces identical predictions.")

## 6. Latency benchmarks

Rough online-inference numbers. In practice, batching is a huge lever — `predict_batch` avoids per-call Python/Rust boundary overhead.

In [ ]:
# --- single predict latency (p50 / p95) ---
single_latencies_ms = []
for uid in RNG.choice(user_ids, size=200, replace=False):
    inter = dict(df_interactions.filter(pl.col("user_id") == uid).select(["item_id", "value"]).iter_rows())
    feats = dict(df_user_features.filter(pl.col("user_id") == uid).select(["feature_name", "value"]).iter_rows())
    t0 = time.perf_counter()
    loaded.predict(inter, feats, top_k=20)
    single_latencies_ms.append((time.perf_counter() - t0) * 1000)

single_latencies_ms.sort()
p50 = single_latencies_ms[len(single_latencies_ms)//2]
p95 = single_latencies_ms[int(len(single_latencies_ms)*0.95)]
p99 = single_latencies_ms[int(len(single_latencies_ms)*0.99)]
print(f"Single predict: p50={p50:.2f}ms  p95={p95:.2f}ms  p99={p99:.2f}ms  mean={statistics.mean(single_latencies_ms):.2f}ms")

# --- batch predict throughput ---
batch_users = []
for uid in user_ids[:500]:
    batch_users.append({
        "interactions": dict(df_interactions.filter(pl.col("user_id") == uid).select(["item_id", "value"]).iter_rows()),
        "features": dict(df_user_features.filter(pl.col("user_id") == uid).select(["feature_name", "value"]).iter_rows()),
    })

t0 = time.perf_counter()
_ = loaded.predict_batch(batch_users, top_k=20)
batch_s = time.perf_counter() - t0
print(f"Batch predict:  {len(batch_users)} users in {batch_s*1000:.1f}ms  ({len(batch_users)/batch_s:.0f} users/sec)")

## 7. Qualitative testing — persona cards

Pick a handful of synthetic personas that represent real user segments. For each, print the top-10 recommendations and annotate with the item's primary genre. The goal is a sanity check: does a "Free Mobile LATAM action fan" actually see action/sci-fi titles?

In [ ]:
personas = [
    {
        "name": "Premium Mobile US power user (action fan)",
        "interactions": {"I0000": 4.5, "I0005": 3.8, "I0015": 4.1},  # all Action-genre items by construction (idx % 5 == 0)
        "features": {"plan_Premium": 1.0, "device_Mobile": 1.0, "region_US": 1.0},
    },
    {
        "name": "Cold-start Free Console APAC",
        "interactions": {},
        "features": {"plan_Free": 1.0, "device_Console": 1.0, "region_APAC": 1.0},
    },
    {
        "name": "Cold-start Premium TV EMEA",
        "interactions": {},
        "features": {"plan_Premium": 1.0, "device_TV": 1.0, "region_EMEA": 1.0},
    },
    {
        "name": "Totally anonymous (no interactions, no features)",
        "interactions": {},
        "features": {},
    },
]

for p in personas:
    recs = loaded.predict(p["interactions"], p["features"], top_k=10)
    genre_hist = Counter(item_genre[g] for g, _ in recs)
    print(f"\n{p['name']}")
    print(f"  top-10 genre histogram: {dict(genre_hist)}")
    for guid, score in recs:
        print(f"  {guid}  genre={item_genre[guid]:<12s}  type={item_type[guid]:<7s}  score={score:.4f}")

## 8. Qualitative testing — More-Like-This coherence

For a random sample of items, print their top-5 neighbors and highlight whether the neighbor's primary genre matches the seed. High genre agreement = the model has learned a coherent item-item signal.

In [ ]:
sample_seeds = list(RNG.choice(item_ids, size=8, replace=False))
genre_match_count = 0
total_neighbors = 0

for seed in sample_seeds:
    neighbors = loaded.predict_similar_items(seed, top_k=5)
    print(f"\n{seed} (genre={item_genre[seed]}, type={item_type[seed]}):")
    for nid, score in neighbors:
        match = "*" if item_genre[nid] == item_genre[seed] else " "
        print(f"  {match} {nid}  genre={item_genre[nid]:<12s}  score={score:.4f}")
        if item_genre[nid] == item_genre[seed]:
            genre_match_count += 1
        total_neighbors += 1

print(f"\nGenre-match rate across neighbors: {genre_match_count}/{total_neighbors} = {genre_match_count/total_neighbors:.1%}")

## Deployment checklist

Before promoting the artifact:

- [x] `fease.validate_data` passed against historical baselines.
- [x] `model.validate()` passed after training.
- [x] Save/reload round-trip produces bit-identical predictions.
- [x] Latency p95 is within the service's SLO.
- [x] Persona spot-checks look sensible (expected genre mixes).
- [x] MLT genre-match rate above a chosen floor (e.g. 60%).
- [x] Manifest captures hyperparameters, data shape, and timestamp.

Copy the `.fease` file and `manifest.json` together to your model store. Downstream services should refuse to load a model whose manifest doesn't match expected dimensions or minimum validation_passed=true.